<a href="https://colab.research.google.com/github/Olamyy/pale/blob/hash-cache-no-op-path/notebooks/pytorch_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pale — PyTorch transfer learning cross-run benchmark

Tests Pale's cross-run deduplication for the transfer learning scenario:
a single pretrained ResNet-18 base is fine-tuned in N independent experiments.
All runs share the same frozen base layer weights — Pale should store those
chunks exactly once, regardless of how many runs reference them.

**Setup:**
- Base: `ResNet18_Weights.DEFAULT` (pretrained ImageNet weights, ~45 MB)
- Fine-tuning: 4 independent runs with different seeds and learning rates
- Each run: 10 epochs, FC layer only (backbone fully frozen throughout)

**Key questions:**
1. What fraction of chunks are shared across fine-tuning runs?
2. How much does the second/third/fourth run cost in new bytes written?
3. What is the no-op rate within each run?
4. How does Pale storage compare to DVC across all 4 runs combined?

In [ ]:
!pip install -q git+https://github.com/Olamyy/pale.git@hash-cache-no-op-path zstandard torch torchvision safetensors dvc

In [ ]:
import subprocess
import sys
import time
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from safetensors.torch import save_file as safetensors_save

sys.path.insert(0, str(Path(".").resolve()))
from utils import (
    CHUNK_SIZE,
    extract_pytorch,
    measure_noop,
    measure_chunk_dedup,
    measure_crossrun,
    pale_bytes,
    print_noop,
    print_chunk,
    print_crossrun,
    _fmt_bytes,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("Imports OK")

## Configuration

In [ ]:
def dvc_init(repo_dir: Path, remote_dir: Path) -> None:
    repo_dir.mkdir(parents=True, exist_ok=True)
    remote_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", "bench@pale"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.name", "Pale Bench"], cwd=repo_dir, check=True)
    subprocess.run(["dvc", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(
        ["dvc", "remote", "add", "-d", "local", str(remote_dir)],
        cwd=repo_dir, check=True,
    )


def dvc_track_and_push(repo_dir: Path, checkpoint_path: Path) -> None:
    subprocess.run(["dvc", "add", str(checkpoint_path)], cwd=repo_dir, check=True,
                   capture_output=True)
    subprocess.run(["dvc", "push"], cwd=repo_dir, check=True, capture_output=True)


def dvc_cache_bytes(remote_dir: Path) -> int:
    return sum(p.stat().st_size for p in remote_dir.rglob("*") if p.is_file())


print("DVC helpers OK")

In [ ]:
N_EPOCHS = 10
NUM_CLASSES = 10

# Four fine-tuning runs: vary seed and learning rate to simulate real experiments
RUNS = [
    {"run_id": "run_a", "seed": 42,  "lr": 1e-3},
    {"run_id": "run_b", "seed": 99,  "lr": 1e-3},
    {"run_id": "run_c", "seed": 7,   "lr": 5e-4},
    {"run_id": "run_d", "seed": 123, "lr": 2e-3},
]

print(f"Epochs per run: {N_EPOCHS}")
print(f"Runs: {len(RUNS)}")
print(f"Total checkpoints: {N_EPOCHS * len(RUNS)}")

## Model setup

Uses `ResNet18_Weights.DEFAULT` (pretrained ImageNet weights). The FC layer is
replaced to match `NUM_CLASSES`. The backbone is fully frozen from epoch 1 —
only the FC layer trains. This is the canonical transfer learning setup:
pretrained base + task-specific head.

All four runs start from the same pretrained weights. The only things that differ
across runs are the FC layer initialization (via seed) and the learning rate.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

def _make_pretrained_resnet18(num_classes: int = NUM_CLASSES) -> nn.Module:
    """Load pretrained ResNet-18, replace FC, freeze backbone."""
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    # Freeze everything except the FC layer
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad_(False)
    return model.to(device)


def _count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Inspect the model before training
sample = _make_pretrained_resnet18()
total, trainable = _count_params(sample)
state = sample.state_dict()
tensor_bytes = sum(v.numel() * v.element_size() for v in state.values())
print(f"ResNet-18 (pretrained, {NUM_CLASSES} classes)")
print(f"  Parameters : {total:,} total, {trainable:,} trainable ({100*trainable/total:.2f}%)")
print(f"  State dict : {len(state)} tensors, {_fmt_bytes(tensor_bytes)} uncompressed")
print(f"  Frozen     : {total - trainable:,} parameters (entire backbone)")
del sample

## Training

Each run fine-tunes a fresh copy of the pretrained ResNet-18 with its own seed
and learning rate. The backbone is frozen throughout — only the FC layer updates.

Synthetic 32×32 images are used (no ImageNet download required).

In [ ]:
def _make_dataset(seed: int):
    rng = np.random.default_rng(seed)
    X = torch.from_numpy(rng.standard_normal((512, 3, 32, 32)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, NUM_CLASSES, 512).astype(np.int64)).to(device)
    return torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True, num_workers=0,
    )


def fine_tune(
    seed: int,
    lr: float,
    n_epochs: int = N_EPOCHS,
) -> list[dict]:
    """Fine-tune pretrained ResNet-18 (frozen backbone, FC only).
    Returns one state dict per epoch.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = _make_pretrained_resnet18()
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss()
    loader = _make_dataset(seed)

    state_dicts = []
    model.train()
    for epoch in range(1, n_epochs + 1):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})

    return state_dicts


print("Training helpers OK")

In [ ]:
all_state_dicts = {}

for run in RUNS:
    t0 = time.time()
    print(f"Training {run['run_id']} (seed={run['seed']}, lr={run['lr']})...", end=" ", flush=True)
    all_state_dicts[run["run_id"]] = fine_tune(seed=run["seed"], lr=run["lr"])
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints trained: {sum(len(v) for v in all_state_dicts.values())}")

## Extract tensors

In [ ]:
all_seqs = {
    run_id: [extract_pytorch(sd) for sd in sds]
    for run_id, sds in all_state_dicts.items()
}

sample_seq = all_seqs["run_a"]
n_tensors = len(sample_seq[0])
total_bytes = sum(v.nbytes for v in sample_seq[-1].values())
print(f"Tensors per checkpoint : {n_tensors}")
print(f"Tensor bytes (run_a epoch 10) : {_fmt_bytes(total_bytes)}")

## No-op fast path — within each run

With the backbone fully frozen from epoch 1, we expect:
- Conv weights, BN weight/bias: 100% no-op (frozen parameters, never updated)
- BN running stats: 0% no-op (updated every forward pass in train mode)
- FC weight/bias: 0% no-op (the only trainable layer)

All four runs should show the same pattern since the frozen set is identical.

In [ ]:
for run_id, seqs in all_seqs.items():
    noop_stats = measure_noop(seqs)
    overall = noop_stats["__summary__"]["identical_pct"]
    print(f"{run_id}: {overall:.1f}% overall no-op")

print()
# Detailed breakdown for run_a
print_noop("run_a (detailed)", measure_noop(all_seqs["run_a"]), max_tensors=20)

### Results (Colab, March 2026)

**No-op fast path — 49.2% overall, identical across all 4 runs:**

All four runs show exactly 49.2% — because the frozen set is identical (same pretrained backbone) and BN running stats update at the same rate regardless of seed or learning rate.

| Tensor group | No-op | Reason |
|---|---|---|
| Conv weights (all layers) | 100% | Frozen from epoch 1 — never updated |
| BN weight / bias | 100% | Frozen from epoch 1 — never updated |
| BN `num_batches_tracked` | 0% | Incremented every forward pass |
| BN `running_mean` / `running_var` | 0% | Updated every forward pass |
| FC weight / bias | 0% | Only trainable layer — updates every epoch |

49.2% is slightly higher than the frozen-backbone result (38.8%) from `pytorch_resnet.ipynb` because here the backbone is frozen from epoch 1 instead of epoch 5 — all 9 consecutive pairs show frozen conv weights, vs only 14 of 19 in the earlier experiment.

## Cross-run chunk sharing

This is the core experiment. All four runs start from the same pretrained weights.
The frozen backbone layers (conv weights, BN weight/bias) are identical across
all runs — they never update. Only the FC layer and BN running stats differ.

Expected:
- High sharing (~80–90%) between any two runs: the backbone chunks are shared
- Sharing does not degrade with more runs: each additional run writes only FC + BN stats

We measure pairwise (run_a vs each other run) and then cumulative
(how many new chunks does each successive run add to the store).

In [ ]:
print("Pairwise cross-run chunk sharing vs run_a:\n")
run_ids = list(all_seqs.keys())
for run_id in run_ids[1:]:
    stats = measure_crossrun(all_seqs["run_a"], all_seqs[run_id], CHUNK_SIZE)
    print_crossrun(f"run_a vs {run_id}", stats)

In [ ]:
# Cumulative: how many new unique chunks does each run add to the store?
# Simulates a shared PaleStore where all runs write into the same CAS.

from pale.hashing import hash_chunk as _hash
from pale.chunking import chunk_bytes as _chunk
from pale.serialization import tensor_to_bytes as _tensor_to_bytes

def _unique_hashes(seqs) -> set:
    s = set()
    for tensors in seqs:
        for arr in tensors.values():
            raw, _, _ = _tensor_to_bytes(arr)
            for c in _chunk(raw, CHUNK_SIZE):
                s.add(_hash(c))
    return s


print(f"{'Run':<8} {'New chunks':>12} {'Cumulative':>12} {'New bytes (est)':>16} {'Marginal cost':>14}")
print("=" * 66)

cumulative: set = set()
for run in RUNS:
    run_id = run["run_id"]
    hashes = _unique_hashes(all_seqs[run_id])
    new = hashes - cumulative
    cumulative |= hashes
    new_bytes_est = len(new) * CHUNK_SIZE
    marginal_pct = len(new) / len(hashes) * 100 if hashes else 0
    print(f"  {run_id:<6} {len(new):>12,} {len(cumulative):>12,} {_fmt_bytes(new_bytes_est):>16} {marginal_pct:>13.1f}%")

### Results (Colab, March 2026)

**Pairwise cross-run sharing — 34.9% (225 of 645 chunks) between any two runs:**

```
[run_a vs run_b]  Shared: 225 / 645  (34.9%)
[run_a vs run_c]  Shared: 225 / 645  (34.9%)
[run_a vs run_d]  Shared: 225 / 645  (34.9%)
```

Identical across all pairs — the 225 shared chunks are the frozen backbone: conv weights and BN weight/bias. These are byte-identical across all runs because they come from the same pretrained checkpoint and are never updated. The remaining 420 unique chunks per run are BN running stats (which diverge immediately from epoch 1 since each run uses different data augmentation seeds) and FC weights (which diverge due to different seeds and learning rates).

**Cumulative new chunks per run:**

```
Run        New chunks   Cumulative  New bytes (est)  Marginal cost
==================================================================
  run_a           645          645          161.2MB         100.0%
  run_b           420        1,065          105.0MB          65.1%
  run_c           420        1,485          105.0MB          65.1%
  run_d           420        1,905          105.0MB          65.1%
```

Each run after the first adds only 420 new chunks (65.1%) — the frozen backbone (225 chunks) is already in the store from run_a and costs nothing to "add" again. The 420 new chunks per run are FC weights + BN running stats, which are unique to each run.

## Pale shared store — actual bytes written

Saves all four runs into a single `PaleStore` instance (shared CAS root).
Measures how many bytes are written to disk after each run is added.

This is the real storage cost, not an estimate: actual `.chunk` file sizes
on disk after zstd compression.

---
## 4-way storage comparison — all 4 runs combined

Compares all four storage methods across the full 4-run × 10-epoch experiment:

| Method | What it does |
|---|---|
| `torch.save` | Naive full save — one `.pt` file per checkpoint, no dedup |
| safetensors | Same full save, HF safetensors format instead of pickle-based `.pt` |
| DVC (real) | File-level versioning — `dvc add` + `dvc push` to a local remote |
| Pale | Tensor-level dedup — frozen backbone stored once across all runs |

DVC operates on the same `.pt` files as `torch.save` — it adds versioning on top
but does not change the per-file storage format. For this reason DVC savings over
`torch.save` reflect only file-level dedup (none, since each checkpoint is unique).
Pale savings come from tensor-level dedup across runs.

In [ ]:
from pale.store import PaleStore
from pale.adapters.pytorch import PyTorchAdapter


def _sd_to_model(sd):
    m = _make_pretrained_resnet18()
    m.load_state_dict({k: v.clone() for k, v in sd.items()})
    return m


with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo      = tmp / "dvc_repo"
    dvc_remote    = tmp / "dvc_remote"
    pt_dir        = dvc_repo / "pt"      # inside the repo so dvc add works
    st_dir        = tmp / "safetensors"
    pale_root     = tmp / "pale"

    pt_dir.mkdir(parents=True)
    st_dir.mkdir()
    dvc_init(dvc_repo, dvc_remote)

    pt_files = []
    st_files = []
    pale_size_after = {}

    for run in RUNS:
        run_id = run["run_id"]
        run_pt = pt_dir / run_id;  run_pt.mkdir()
        run_st = st_dir / run_id;  run_st.mkdir()

        for epoch, sd in enumerate(all_state_dicts[run_id], 1):
            # torch.save (inside dvc_repo so dvc add works)
            pt_path = run_pt / f"epoch_{epoch:02d}.pt"
            torch.save(sd, pt_path)
            pt_files.append(pt_path)

            # safetensors — requires contiguous float/int tensors
            st_path = run_st / f"epoch_{epoch:02d}.safetensors"
            safetensors_save({k: v.contiguous() for k, v in sd.items()}, st_path)
            st_files.append(st_path)

            # DVC tracks the .pt files
            dvc_track_and_push(dvc_repo, pt_path)

        # Pale
        with PaleStore(root=pale_root, run_id=run_id, adapter=PyTorchAdapter()) as store:
            for epoch, sd in enumerate(all_state_dicts[run_id], 1):
                store.save(_sd_to_model(sd), step=epoch)

        pale_size_after[run_id] = pale_bytes(pale_root)

    torch_b = sum(p.stat().st_size for p in pt_files)
    st_b    = sum(p.stat().st_size for p in st_files)
    dvc_b   = dvc_cache_bytes(dvc_remote)
    pale_b  = pale_bytes(pale_root)

def _savings(base, b):
    return (base - b) / base * 100 if base else 0

n_total = len(RUNS) * N_EPOCHS
print(f"4-way storage comparison — {len(RUNS)} runs × {N_EPOCHS} epochs ({n_total} checkpoints total)\n")
print(f"  {'Method':<20} {'Total bytes':>12} {'vs torch.save':>14}")
print(f"  {'-'*20} {'-'*12} {'-'*14}")
print(f"  {'torch.save':<20} {_fmt_bytes(torch_b):>12} {'—':>14}")
print(f"  {'safetensors':<20} {_fmt_bytes(st_b):>12} {_savings(torch_b, st_b):>13.1f}%")
print(f"  {'DVC (real)':<20} {_fmt_bytes(dvc_b):>12} {_savings(torch_b, dvc_b):>13.1f}%")
print(f"  {'Pale':<20} {_fmt_bytes(pale_b):>12} {_savings(torch_b, pale_b):>13.1f}%")

print(f"\nPale marginal cost per run:")
prev = 0
for run_id, size in pale_size_after.items():
    print(f"  {run_id:<8} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

### Results (Colab, March 2026)

```
4-way storage comparison — 4 runs × 10 epochs (40 checkpoints total)

  Method               Total bytes   vs torch.save
  -------------------- ------------ --------------
  torch.save                  1.7GB              —
  safetensors                 1.7GB           0.1%
  DVC (real)                  1.7GB           0.0%
  Pale                       39.7MB          97.7%

Pale marginal cost per run:
  run_a      39.6MB  (+39.6MB)
  run_b      39.6MB  (+53.6KB)
  run_c      39.7MB  (+53.6KB)
  run_d      39.7MB  (+53.6KB)
```

**Layer sharing:** All conv weights and BN weight/bias tensors are 100% shared across all 4 runs (frozen backbone — never updated). All BN `running_mean`, `running_var`, `num_batches_tracked`, FC weight, and FC bias are 0% shared (40 unique versions each — one per epoch per run).

## Which layers are shared and which are unique per run?

Breaks down chunk sharing by layer group across all four runs.
Frozen backbone layers should show 100% sharing; FC and BN running stats should show 0%.

In [ ]:
# For each tensor name, collect all unique chunk hashes across all runs and all steps
from collections import defaultdict

tensor_hashes_per_run: dict[str, dict[str, set]] = defaultdict(lambda: defaultdict(set))

for run_id, seqs in all_seqs.items():
    for tensors in seqs:
        for name, arr in tensors.items():
            raw, _, _ = _tensor_to_bytes(arr)
            for c in _chunk(raw, CHUNK_SIZE):
                tensor_hashes_per_run[name][run_id].add(_hash(c))

# For each tensor: are its chunks identical across all 4 runs?
tensor_names = sorted(tensor_hashes_per_run.keys())
run_ids = [r["run_id"] for r in RUNS]

print(f"  {'Tensor':<45} {'Shared across all runs':>22} {'Per-run unique':>15}")
print(f"  {'-'*45} {'-'*22} {'-'*15}")

for name in tensor_names:
    per_run = tensor_hashes_per_run[name]
    all_hashes = set.union(*per_run.values())
    shared = set.intersection(*per_run.values())
    shared_pct = len(shared) / len(all_hashes) * 100 if all_hashes else 0
    unique_per_run = len(all_hashes) - len(shared)
    print(f"  {name:<45} {shared_pct:>20.1f}% {unique_per_run:>15,}")

## Summary

| Question | Result |
|---|---|
| No-op rate within each run | 49.2% — conv weights + BN params 100%; BN running stats + FC 0% |
| Chunk sharing run_a vs any other run | 34.9% (225 of 645 chunks) — the entire frozen backbone |
| Marginal cost of run 2, 3, 4 | ~53 KB each — 0.1% of the first run |
| Which layers are shared | All conv weights, all BN weight/bias (100% across all runs) |
| Which layers are unique per run | FC weight/bias, all BN running stats (0% sharing) |

**4-way storage comparison (40 checkpoints across 4 runs):**

| Method | Total bytes | vs torch.save |
|---|---|---|
| torch.save | 1.7 GB | — |
| safetensors | 1.7 GB | 0.1% |
| DVC (real) | 1.7 GB | 0.0% |
| Pale | 39.7 MB | 97.7% |

**Key finding:** `torch.save` and safetensors store full model state every save — no dedup. DVC adds versioning on top of the same files but has no cross-run dedup. Only Pale stores the pretrained backbone once across all runs; every additional fine-tuning run costs only the unique tensors (~53 KB).